# Practical 6: Balance the Dataset for ML/DL Training
**Course:** BE05000231 - PDS (Semester V, GTU GSET)
**Course Outcome:** CO3

### Objective:
Analyze class imbalance in real-world security logs, enforce strict train/test separation to prevent data leakage, and evaluate Random Under-Sampling, Random Over-Sampling, SMOTE, and class weights.

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt

from pipeline.parser import parse_file
from pipeline.cleaner import clean
from pipeline.labeler import load_rules, label_requests
from pipeline.features import add_request_features
from models.balance import compare_balancing_methods, balance_dataset

LOG_PATH = Path('../tests/fixtures/sample_access.log')
RULES_PATH = Path('../rules/web_rules.yaml')


## 2. Ingest, Clean, Label & Extract Features

In [ ]:
raw_df, _ = parse_file(LOG_PATH)
cleaned_df = clean(raw_df)
rules = load_rules(RULES_PATH)
labeled_df = label_requests(cleaned_df, rules)
feat_df = add_request_features(labeled_df)

feature_cols = ['url_length', 'query_length', 'special_chars', 'special_ratio', 'url_entropy', 'digit_ratio']
X = feat_df[feature_cols]
# Binary target for modeling evaluation: benign vs attack
y = (feat_df['label'] != 'benign').astype(int).map({0: 'benign', 1: 'attack'})
print('Class Distribution (Raw Data):')
print(y.value_counts(normalize=True).round(4) * 100)


## 3. Why Accuracy is a Misleading Metric in Cybersecurity
In real production web logs, 99% of requests are benign. A trivial model predicting 'benign' for 100% of cases would achieve 99% accuracy while missing every single intrusion! Therefore, Precision, Recall, F1-score, and PR-AUC must be used.

In [ ]:
comp_df, artifacts = compare_balancing_methods(
    X, y, test_size=0.25, random_state=42,
    output_parquet='../data/processed/train_balanced.parquet'
)
print('Balancing Strategy Comparison Table:')
comp_df


## 4. SMOTE Interpolation Caveat (Viva Critical Insight)
SMOTE generates synthetic samples by interpolating between nearest neighbors in continuous vector space. It is mathematically sound for numeric features (e.g. entropy, lengths), but invalid for raw strings or categorical flags.

In [ ]:
X_train, y_train = artifacts['X_train'], artifacts['y_train']
X_smote, y_smote = balance_dataset(X_train, y_train, method='smote')
print('Before SMOTE: ', dict(y_train.value_counts()))
print('After SMOTE:  ', dict(y_smote.value_counts()))


### Conclusion:
Practical 6 proved the necessity of train-before-resample data hygiene to avoid data leakage, and established that over-sampling and SMOTE significantly improve recall on minority attack vectors.